### Baby script for fixing labels on mismatched trials

In [1]:
import pandas as pd
import numpy as np 
from matplotlib import pyplot as plt
import matplotlib

In [19]:
# load
cols_inc = ["Subject","Nap_ID","Trigger", "Expected_Muscle","Nb_Corr","Nb_Zygo","Is_Correct"] 
true_labels = pd.read_excel("final_scores_16042026_v3.xlsx")

# files to relabel 
features_all = pd.read_pickle("training_features_19032026.pkl")
features_all = features_all.rename(columns={'Nap Number': 'Nap'})

trial_info = pd.read_csv("Trial_information_narcolepsy.csv", usecols=cols_inc) 

In [11]:
true_labels.head()

,Subject,Nap,Triggers_Order_Nap,Epoch,Muscle_type,Response_start_sample,Response_end_sample,Contraction_number
0,NL03JV,1,30,1.0,Corr,157.0,1943.0,0.0
1,NL03JV,1,30,1.0,Zygo,222.0,736.0,3.0
2,NL03JV,1,40,2.0,Corr,12.0,2233.0,0.0
3,NL03JV,1,40,2.0,Zygo,1449.0,1739.0,1.0
4,NL03JV,1,50,3.0,Corr,47.0,2188.0,0.0


Relabeling the files

In [16]:
key_cols = ['Subject', 'Nap', 'Triggers_Order_Nap']

true_labels_wide = (
    true_labels[true_labels['Muscle_type'].isin(['Zygo', 'Corr'])]
    .pivot_table(
        index=key_cols,
        columns='Muscle_type',
        values='Contraction_number',
        aggfunc='first'   # or 'max' if duplicates exist and you want that behavior
    )
    .reset_index()
    .rename(columns={
        'Zygo': 'Num_Contractions_Zygo',
        'Corr': 'Num_Contractions_Corr'
    })
)


In [18]:
features_all.head()

,Subject,nap,Triggers_Order_Nap,True_Muscle_Activated,Num_Contractions_Zygo,Num_Contractions_Corr,WL_Zygo,Var_Zygo,RMS_Zygo,MAVS_Zygo,WL_Corr,Var_Corr,RMS_Corr,MAVS_Corr,Zygo,Corr
0,RL19RS,1,1,Corr,0,3,"[16.979501139871267, 18.165877955842884, 18.59...","[2.4357275001701533, 2.401023751914199, 2.3740...","[2.063547532557064, 2.0308865582680564, 1.9948...","[-0.037187916340846394, -0.04570457765223557, ...","[75.29678021851888, 76.39875013161338, 78.5484...","[3.0509425198291757, 3.047810217522341, 3.1377...","[1.7913263414331733, 1.7885089584884002, 1.803...","[-0.008847728745045247, 0.020195338894544834, ...","[-2.7277040853500525, 0.31356623348522716, 0.8...","[-0.7911568335676051, 1.912198159234712, -0.86..."
1,RL19RS,1,2,Corr,0,3,"[26.824654542616372, 27.632174893961363, 27.76...","[0.8344136524953758, 0.8463100634295447, 0.862...","[0.9188357871709856, 0.927154584528815, 0.9381...","[0.016124978477620067, 0.018820054425093158, 0...","[88.84586351699936, 91.72494689471449, 94.9541...","[5.900218213130837, 6.0033229929637, 6.0065618...","[2.549990265745516, 2.551919890588013, 2.55310...","[0.003144624495356796, 0.0019711226458358766, ...","[-0.07304423025304638, -2.2181182276310256, -2...","[1.4867179586649941, 3.038561724391297, 0.4844..."
2,RL19RS,1,3,Zygo,3,0,"[24.883849912053638, 25.700508948621653, 26.33...","[0.7468896743468332, 0.7497155323785327, 0.770...","[0.8650623074114399, 0.8669707886369807, 0.879...","[0.005861488405386139, 0.0186177487040049, 0.0...","[111.49377377841105, 112.30307576087087, 114.3...","[8.66934942006402, 8.668903914616124, 8.716578...","[2.9725897038242906, 2.970058549628821, 2.9811...","[-0.01646558841278223, 0.02292561840954077, 0....","[0.13543491172318878, -1.0559624223971724, -0....","[-0.8684031943153836, 0.22778864107415986, -0...."
3,RL19RS,1,4,Zygo,3,0,"[24.477820981376315, 26.8812398019832, 27.3165...","[0.44507661010216365, 0.5037639088652561, 0.53...","[0.7626033518594519, 0.813616368572243, 0.8420...","[0.0283114732007127, 0.019604390522114823, 0.0...","[105.15945939835385, 113.62873732491099, 118.6...","[11.301421260842174, 11.568401993181608, 11.59...","[3.392698218078343, 3.4405619281245943, 3.4388...","[0.06157093211621323, -0.00626023389568342, 0....","[-0.7122686077788487, 0.11544473391500598, 0.2...","[1.1167298715401666, 1.5394483862679575, -0.22..."
4,RL19RS,1,5,Corr,0,3,"[23.771646650420337, 25.250403446319936, 26.60...","[1.8706616369307074, 1.8540474174075106, 1.902...","[1.7844795652915169, 1.7573151489189478, 1.732...","[-0.03526536750283582, -0.027489667134452667, ...","[97.88042157234477, 107.45667002602518, 110.58...","[8.045422533890312, 8.250081346269646, 8.24998...","[2.8438684580292426, 2.88469073194194, 2.88459...","[0.061789720889836586, -0.00080370477739522, 0...","[-2.245731503659271, -1.7656176164993092, 0.67...","[-0.34758477119918696, -2.500891459504272, -0...."


In [21]:
true_labels_wide.head()
df_all = features_all.merge(true_labels_wide, on=key_cols, how='left')

zygo_mismatch = (
    df_all['Num_Contractions_Zygo'].notna() &
    (df_all['Num_Contractions_Zygo'] != df_all['Contraction_number_Zygo'])
)

corr_mismatch = (
    df_all['Contraction_number_Corr'].notna() &
    (df_all['Num_Contractions_Corr'] != df_all['Contraction_number_Corr'])
)

df_all.loc[zygo_mismatch, 'Num_Contractions_Zygo'] = df_all.loc[zygo_mismatch, 'Num_Contractions_Zygo']
df_all.loc[corr_mismatch, 'Num_Contractions_Corr'] = df_all.loc[corr_mismatch, 'Contraction_number_Corr']

# --- Optional: see what changed ---
changed_rows = df_all[zygo_mismatch | corr_mismatch].copy()
print("Number of rows updated:", len(changed_rows))
print(changed_rows[
    key_cols +
    ['Num_Contractions_Zygo', 'Num_Contractions_Corr',
     'Num_Contractions_Zygo', 'Contraction_number_Corr']
].head())


KeyError: 'Contraction_number_Zygo'